# C2.2 / C2.3 - Teacher labels and plain-English rewrites (FinSight, Colab)

Runs Qwen3 AWQ teachers with **vLLM** over a risks file (`risk_id`, `title`, `body`). Inference
only. One JSON answer per risk: `category`, `seriousness_1to5`, `hard_fact`, `simple`
(<= 60 words), `numbers_copied`. Thinking mode is off. The prompt and the loop are copied from
`finsight.risks` by `scripts/make_colab_notebooks.py`, so they are the tested ones.

- **Bake-off (C2.2):** `JOB = "teacher_bakeoff"`, both models, A100 (the 32B model needs ~40 GB).
- **Full run (C2.3):** `JOB = "teacher_full"`, one model, `EVERY = 200`.
- **Smoke first** (`SMOKE = True`, 20 risks) on a T4 or L4 with the small model; then `SMOKE = False`.
- Answers are appended to local disk and copied to Drive after every batch; a disconnect resumes.
- Filters run on the laptop, not here. Teacher outputs are AI labels (`label_source`).

In [ ]:
# ---- parameters (edit, then Run all) ------------------------------------------------------
JOB = "teacher_bakeoff"            # Drive folder MyDrive/FinSight/<JOB>/ ; "teacher_full" for the real run
INPUT_FILE = "bakeoff_risks.jsonl" # in that folder: risk_id, title, body (risks.jsonl for the full run)
MODELS = ["Qwen/Qwen3-14B-AWQ", "Qwen/Qwen3-32B-AWQ"]   # full run: only the chosen model
SMOKE = True                       # True: 20 risks per model, then stop (run on a T4/L4 first)
EVERY = 100                        # checkpoint interval (risks per vLLM batch); 200 for the full run
MAX_MODEL_LEN = 4096
MAX_TOKENS = 400
VLLM_PIN = "vllm==0.11.0"          # REPLACE with the pin recorded by the L4 rate check (COLAB_STEPS_rate_check)
QUANT = "awq_marlin"               # awq_marlin needs sm80+ (L4/A100); a T4 needs "awq"
GPU_MEM = 0.90
MIN_GB_FOR_32B = 40                # the 32B model is skipped on a smaller GPU (L4-only fallback, C03 section 3)
UNITS_BEFORE = None                # compute units in the Resources panel before the run
UNITS_AFTER = None                 # ... and after (type it, then re-run the last cell)
UNASSIGN = False                   # True disconnects the runtime at the end

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", VLLM_PIN], check=True)

In [ ]:
# Shared helpers: copy notebooks/colab/_common.py to MyDrive/FinSight/_common.py once (COLAB_STEPS).
import sys, time
sys.path.insert(0, "/content/drive/MyDrive/FinSight")
from google.colab import drive
drive.mount("/content/drive")
from _common import mount_drive, Checkpointer, run_summary, gpu_info, unassign_runtime
STARTED = time.time()
print(gpu_info())

In [ ]:
"""Resumable teacher run (B03 §3.4): raw answers appended to a JSONL checkpoint in batches.

Standard library only: ``scripts/make_teacher_notebook.py`` copies this file into the Kaggle
notebook, so the loop tested here is the loop that runs on the GPU. A rerun skips every
``risk_id`` already in the output file (the resume flag is simply running it again).
"""

from __future__ import annotations

import json
import os
from collections.abc import Callable, Iterable, Iterator
from pathlib import Path
from typing import Any

Generate = Callable[[list[list[dict[str, str]]]], list[str]]  # chats in, raw texts out


def read_jsonl(path: Path) -> Iterator[dict[str, Any]]:
    """Rows of a JSON Lines file (nothing if the file is missing)."""
    if not path.exists():
        return
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except json.JSONDecodeError:
                continue  # a line cut short by a stopped session; that risk is redone


def done_ids(path: Path) -> set[str]:
    """Risk ids already answered, so a resumed run skips them."""
    return {str(row["risk_id"]) for row in read_jsonl(path) if "risk_id" in row}


def _append(path: Path, rows: list[dict[str, Any]]) -> None:
    cut = False
    if path.exists() and path.stat().st_size > 0:
        with path.open("rb") as f:
            f.seek(-1, os.SEEK_END)
            cut = f.read(1) != b"\n"
    with path.open("a", encoding="utf-8") as f:
        if cut:  # keep the stopped session's half line on its own, so it stays unreadable
            f.write("\n")
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
        f.flush()
        os.fsync(f.fileno())


def run_teacher(
    items: Iterable[dict[str, Any]],
    generate: Generate,
    out_path: Path,
    *,
    every: int = 100,
    limit: int | None = None,
    meta: dict[str, Any] | None = None,
    after_batch: Callable[[], None] | None = None,
) -> dict[str, int]:
    """Send each item's ``messages`` to ``generate`` in batches of ``every`` and checkpoint.

    Args:
        items: dicts with ``risk_id`` and ``messages`` (chat format).
        generate: one call per batch; returns one raw text per chat, in order.
        out_path: JSONL checkpoint; rows are ``{risk_id, raw, **meta}``.
        every: batch size, and so the checkpoint interval.
        limit: stop after this many new items (pilot and smoke runs).
        meta: copied into every row (model, prompt_version).
        after_batch: called after every checkpoint write (Colab: copy the file to Drive).

    Returns:
        ``{"skipped": already done, "written": new rows}``.
    """
    out_path.parent.mkdir(parents=True, exist_ok=True)
    done = done_ids(out_path)
    skipped = written = 0
    batch: list[dict[str, Any]] = []

    def flush() -> None:
        nonlocal written
        if not batch:
            return
        raws = generate([it["messages"] for it in batch])
        if len(raws) != len(batch):
            raise ValueError(f"generate returned {len(raws)} answers for {len(batch)} chats")
        _append(
            out_path,
            [
                {"risk_id": it["risk_id"], "raw": r, **(meta or {})}
                for it, r in zip(batch, raws, strict=True)
            ],
        )
        written += len(batch)
        print(f"checkpoint: {written} new rows in {out_path.name}", flush=True)
        batch.clear()
        if after_batch is not None:
            after_batch()

    for item in items:
        rid = str(item["risk_id"])
        if rid in done:
            skipped += 1
            continue
        if limit is not None and written + len(batch) >= limit:
            break
        done.add(rid)
        batch.append(item)
        if len(batch) >= every:
            flush()
    flush()
    return {"skipped": skipped, "written": written}

In [ ]:
import json

# GENERATED from finsight.risks.teacher by scripts/make_teacher_notebook.py; do not edit
PROMPT_VERSION = 'teacher-v1'
SYSTEM = 'You label risk factors from Indian IPO offer documents and rewrite them in plain\nEnglish. Answer with one JSON object and nothing else.\n\nFields:\n- "category": exactly one of financial, debt_liquidity, customers_suppliers, competition, legal_litigation, regulatory, promoters_governance, operations, technology_data, market_macro.\n- "seriousness_1to5": 1 = routine boilerplate, 5 = a specific, material problem the company\n  already has.\n- "hard_fact": true if the risk describes something that has already happened (a loss, a case, a\n  default), false if it only describes something that may happen.\n- "simple": the risk in plain English for a first-time investor, at most 60 words, sentences under\n  20 words. Keep every number exactly as written, with its unit. Keep how certain it is: "may" stays\n  "may", "has" stays "has". Add no facts. Never tell anyone to buy, sell, apply, invest or avoid,\n  and never judge the IPO as good, bad, safe or worth it.\n- "numbers_copied": every number you used in "simple", copied exactly.\n\nCategories: financial = losses, falling revenue or margins, cash flow, working capital; debt_liquidity = borrowings, repayment, covenants, liquidity, interest rates on our loans; customers_suppliers = dependence on few customers, suppliers or distributors; competition = competitors, pricing pressure, market share; legal_litigation = court cases, claims, penalties, tax disputes; regulatory = laws, licences, approvals, government policy, SEBI/RBI rules; promoters_governance = promoters, related parties, management, control, conflicts of interest; operations = plants, capacity, people, projects, insurance, supply disruptions; technology_data = IT systems, cyber security, data protection, intellectual property; market_macro = economy, industry cycles, currency, share price after listing.'
OUTPUT_SCHEMA = json.loads('{"description": "The JSON the teacher must return for one risk.", "properties": {"category": {"enum": ["financial", "debt_liquidity", "customers_suppliers", "competition", "legal_litigation", "regulatory", "promoters_governance", "operations", "technology_data", "market_macro"], "title": "Category", "type": "string"}, "seriousness_1to5": {"maximum": 5, "minimum": 1, "title": "Seriousness 1To5", "type": "integer"}, "hard_fact": {"title": "Hard Fact", "type": "boolean"}, "simple": {"minLength": 1, "title": "Simple", "type": "string"}, "numbers_copied": {"items": {"type": "string"}, "title": "Numbers Copied", "type": "array"}}, "required": ["category", "seriousness_1to5", "hard_fact", "simple"], "title": "TeacherOutput", "type": "object"}')


def messages(title, body):
    text = f"Title: {title.strip()}\n\n{body.strip()}" if title.strip() else body.strip()
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": text}]

In [ ]:
import gc, json, os, time
from pathlib import Path
import torch
from vllm import LLM, SamplingParams

JOB_DIR = mount_drive(JOB)
LIMIT = 20 if SMOKE else None
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
risks = list(read_jsonl(JOB_DIR / INPUT_FILE))
items = [{"risk_id": r["risk_id"], "messages": messages(r.get("title") or "", r["body"])} for r in risks]
print(len(items), "risks; GPU", gpu_info(), f"{gpu_gb:.0f} GB")
ck = Checkpointer(local_dir="/content/ck", drive_dir=JOB_DIR / "out", every=1)
PINS = {"vllm": VLLM_PIN, "torch": torch.__version__}
for model in MODELS:
    slug = model.split("/")[-1].lower()
    if "32b" in slug and gpu_gb < MIN_GB_FOR_32B:
        print("skipping", model, "- GPU too small; use the 14B model directly (C-ADR-05 fallback)")
        continue
    out_path = ck.path(f"raw_{slug}.jsonl")
    llm = LLM(model=model, quantization=QUANT, dtype="float16", max_model_len=MAX_MODEL_LEN,
              gpu_memory_utilization=GPU_MEM, enforce_eager=False, seed=2026)
    try:  # vLLM <= 0.10
        from vllm.sampling_params import GuidedDecodingParams
        params = SamplingParams(temperature=0.2, top_p=0.9, max_tokens=MAX_TOKENS,
                                guided_decoding=GuidedDecodingParams(json=OUTPUT_SCHEMA))
    except ImportError:  # newer releases renamed it
        from vllm.sampling_params import StructuredOutputsParams
        params = SamplingParams(temperature=0.2, top_p=0.9, max_tokens=MAX_TOKENS,
                                structured_outputs=StructuredOutputsParams(json=OUTPUT_SCHEMA))

    def generate(chats):
        outs = llm.chat(chats, params, use_tqdm=False, chat_template_kwargs={"enable_thinking": False})
        return [o.outputs[0].text for o in outs]

    t0 = time.time()
    stats = run_teacher(items, generate, out_path, every=EVERY, limit=LIMIT,
                        meta={"model": model, "prompt_version": PROMPT_VERSION}, after_batch=ck.sync)
    seconds = round(time.time() - t0)
    ck.sync()
    summary = {**stats, "model": model, "prompt_version": PROMPT_VERSION, "seconds": seconds,
               "smoke": SMOKE, "gpu": gpu_info(), "pins": PINS}
    ck.path(f"run_summary_{slug}.json").write_text(json.dumps(summary, indent=1), encoding="utf-8")
    ck.sync()
    print("TEACHER OK", json.dumps(summary))
    del llm
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
s = run_summary(JOB_DIR, JOB, started=STARTED, units_before=UNITS_BEFORE, units_after=UNITS_AFTER,
                items_done=len(items), pins=PINS, extra={"models": MODELS, "smoke": SMOKE})
print(json.dumps(s, indent=1))
print("Copy raw_*.jsonl, run_summary_*.json (folder out/) and run_summary.json back (COLAB_STEPS).")
unassign_runtime(UNASSIGN)